#### Install requirements

Before the SQL Agent code import the following functions into the coding environment for you:

In [0]:
# %pip install -U langchain-community langchain-databricks langchain
# %pip install databricks-sql-connector
# %pip install databricks-sqlalchemy
# %pip install -U databricks-langchain
# %pip install --force-reinstall --no-cache-dir langchain-core==1.6.13
# %pip install --upgrade --force-reinstall databricks-langchain

In [0]:
# %pip install --upgrade \
#     databricks-langchain \
#     langchain-community \
#     langchain-classic \
#     databricks-sql-connector \
#     databricks-sqlalchemy

#### Check libraries are properly installed or not

In [0]:
import langchain
import langchain_community
import langchain_databricks
import databricks.sql

print("All imports successful!")

---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
File <command-6403840973569583>, line 1
----> 1 import langchain
      2 import langchain_community
      3 import langchain_databricks

ModuleNotFoundError: No module named 'langchain'

In [0]:
# dbutils.library.restartPython()

- Create catalog with name 'catalog_sachin' and schema 'schema_sachin' then upload csv file to make table with name 'housing_price_dataset'.
- Access dataset from this link: https://github.com/sachin365123/top-6-Datasets-for-PowerBI/blob/main/housing_price_dataset.csv

In [0]:
%sql
SHOW CATALOGS;

In [0]:
%sql
SHOW SCHEMAS IN catalog_sachin;

In [0]:
%sql
SHOW TABLES IN catalog_sachin.schema_sachin;

#### Create SQL agent

##### Create LangChain SQL Agent Connected to Databricks
**LangChain is a library to build and deploy LLM applications. It also allows you to create AI agents, connecting connect to data sources and running SQL queries.**

_In the following exercises, you will be interacting with schemas from the samples catalog. In this exercise, you will create a Databricks SQL agent and in subsequent exercises you will also query the agents that you create._

- > Create a LangChain SQLDatabase from the nyctaxi schema in the "samples" Databricks catalog.
- > Create a Databricks LangChain LLM using the Databricks Meta Llama 3 Foundational Model.
- > Create a LangChain SQL Database toolkit that uses the LangChain Databricks SQL db and the llm.
- > Create a LangChain Databricks SQL agent using the toolkit and llm.

In [0]:
from databricks.sdk import WorkspaceClient
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_community.agent_toolkits.sql.base import create_sql_agent
from databricks_langchain import ChatDatabricks

# 1. Initialize Databricks Workspace Client
w = WorkspaceClient()

# 2. Select a SQL warehouse
warehouses = list(w.warehouses.list())

if not warehouses:
    raise RuntimeError("No SQL warehouses found.")

warehouse = warehouses[0]
warehouse_id = warehouse.id

# This code retrieves all Databricks SQL warehouses, raises a `RuntimeError` if none exist, and selects the first warehouse from the list. It then extracts the warehouse’s unique identifier (`warehouse.id`) for use in subsequent SQL operations.

print(f"Warehouse: {warehouse.name}")
print(f"Warehouse ID: {warehouse_id}")
print(f"Warehouse state: {warehouse.state}")

# 3. Connect to your Unity Catalog schema
db = SQLDatabase.from_databricks(
    catalog="catalog_sachin",
    schema="schema_sachin",
    warehouse_id=warehouse_id
)

# 4. Verify your tables
print("Available tables:", db.get_usable_table_names())

# 5. Initialize the Databricks LLM
llm = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    max_tokens=500
)

# 6. Create the SQL toolkit
toolkit = SQLDatabaseToolkit(
    db=db,
    llm=llm
)

# This code initializes a Databricks-hosted Llama 3.3 70B model with a 500-token response limit, then creates a `SQLDatabaseToolkit` that combines the LLM and connected database to help an agent generate and execute SQL queries.

# 7. Create the SQL agent
databricks_sql_agent = create_sql_agent(
    llm=llm,
    toolkit=toolkit,
    verbose=True,
    handle_parsing_errors=True
)

# This code creates a LangChain SQL agent using the Databricks LLM and SQL toolkit, enabling it to interpret natural-language questions, generate and execute SQL queries, and return results. `verbose=True` displays execution steps, while `handle_parsing_errors=True` helps the agent recover from output-parsing errors.

print("SQL agent created successfully!")

#### Query the Databricks SQL Agent.

In [0]:
response = databricks_sql_agent.invoke({
    "input": "Show the first 5 records from housing_price_dataset."
})

print(response["output"])

# This code invokes the LangChain SQL agent with a natural-language request to retrieve the first five records from `housing_price_dataset`. The agent interprets the request, generates and executes the appropriate SQL query, and returns the results in `response`.
